# Week 2: A2A Specialists and MCP Tools

**120-minute lab.** Protocol walkthrough/setup 25 min; service startup and inspection 20 min; three tasks of 15 min each; testing/discussion/buffer 30 min.

Path: notebook coordinator -> official A2A JSON-RPC -> Flight or Hotel -> official MCP -> real Nuitee REST. Only MCP 8010 and the shared A2A process 8001 are needed. No LLM, payment or booking.

Exactly three tasks: Flight's official MCP invocation, Hotel's reuse of W1 normalization/selection, and the coordinator's official A2A message construction. Teacher code handles Agent Cards, servers, connection setup, async lifecycle and artifact parsing.

W1 remains the source of occupancy, prices, selection and budget logic. Read the short SDK examples, edit the named Python function, run the notebook and inspect the returned quotes.

## Setup and Week 1 Gate (25 min)

Use the same Python environment and root `.env` or Codespaces `NUITEE_API_KEY` as Week 1. Teacher setup installs official A2A 1.2.2 and MCP 2.3.0. Complete Week 1 first; no payment SDK is used here.

The next cell reloads `config/trip_scenario.py` and calls the same `get_trip_request()` used in Week 1. Do not create another request or edit notebook constants. If the scenario changes, rerun setup and export the new agent-built plan. Restart the services after changes to running Python modules.

Teacher server registration and task events live in `sdk_support/a2a_server.py`. Official client calls and your message construction remain in `week2/a2a_coordinator.py`. Run the Week 1 acceptance checks before adding protocol boundaries.

In [ ]:
import sys
import subprocess
import importlib
from pathlib import Path
from importlib.metadata import version
from travel_challenge.week1 import trip_planner
from travel_challenge.config import trip_scenario
from travel_challenge.week2 import travel_agents, a2a_coordinator

ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / 'pyproject.toml').exists())
importlib.reload(trip_scenario)
request = trip_planner.get_trip_request()
print({package: version(package) for package in ['a2a-sdk', 'mcp']})
print('Request to both agents:', request)
check = subprocess.run([sys.executable, '-m', 'pytest', 'tests/test_week1.py', '-q'], cwd=ROOT, capture_output=True, text=True)
print(check.stdout)
assert check.returncode == 0, 'Complete Week 1 first. ' + check.stderr

## Tasks 1 and 2: Flight and Hotel (15 min Each)

Edit `src/travel_challenge/week2/travel_agents.py`.

1. `flight_agent`: replace `result = None` with one awaited official `client.call_tool` invocation. Supply tool name `search_flights` and arguments containing the original request. The teacher checks errors, reads `structured_content`, and calls W1 normalization/selection.
2. `hotel_agent`: the official MCP invocation is provided as a worked example with different arguments. Complete the existing return using W1 `normalize_hotels` and `choose_option`. Keep the quote unchanged.

Inspect the tool list and example response in the supplied probe. `await` waits for the remote result; the teacher's `async with Client(...)` opens and closes the connection. Do not call Nuitee directly or reimplement W1 logic.

In [ ]:
print('MCP: Client.call_tool(tool_name, arguments) -> official result')
print('Check result.is_error; result.structured_content is raw provider JSON.')
print('Each agent: MCP response -> W1 normalize -> W1 choose_option -> A2A quote artifact')

In [ ]:
print('A2A: SendMessageRequest(message=...) contains new_text_message(text, role=...).')
print('Serialize the original request with json.dumps; use the user role.')
print('You construct the message. Instructor send_task calls client.send_message(message) and reads the artifact.')
print('Coordinator: two Agent quotes -> W1 make_plan; no async lifecycle code to write.')

## Task 3: Construct an Official A2A Message (15 min)

Edit `src/travel_challenge/week2/a2a_coordinator.py::plan_trip`. Replace `message = None` with an official `SendMessageRequest`, using the helpers shown above and imported in the source file. Preserve the original request as JSON; do not invent quote fields.

Both awaited sends and W1 `make_plan` are supplied. After completing the tasks, inspect both Agent responses and the resulting total below.

Complete source edits before starting services. After editing a running specialist, use the shutdown cell and restart: service subprocesses do not auto-reload. No budget optimizer, replan loop or payment is needed.

In [ ]:
import asyncio
import socket
import tempfile
import time
import httpx

if any(process.poll() is None for process, log in globals().get('services', [])):
    raise RuntimeError('Run the shutdown cell before restarting services.')
for port in [8001, 8010]:
    with socket.socket() as sock:
        assert sock.connect_ex(('127.0.0.1', port)) != 0, f'Port {port} is occupied; stop its owner first.'
services = []
try:
    for module in ['travel_challenge.week2.mcp_tools', 'travel_challenge.week2.travel_agents']:
        log = tempfile.TemporaryFile(mode='w+')
        process = subprocess.Popen([sys.executable, '-m', module], cwd=ROOT, stdout=log, stderr=subprocess.STDOUT)
        services.append((process, log))
    async with httpx.AsyncClient(timeout=2) as client:
        for url in ['http://127.0.0.1:8010/mcp', 'http://127.0.0.1:8001/flights/.well-known/agent-card.json']:
            deadline = time.monotonic() + 30
            while True:
                if any(process.poll() is not None for process, log in services):
                    raise RuntimeError('Service exited; inspect service logs with the shutdown cell.')
                try:
                    async with client.stream('GET', url) as response:
                        if response.status_code in [200, 405, 406]:
                            break
                except httpx.TransportError:
                    pass
                if time.monotonic() > deadline:
                    raise RuntimeError(f'Service startup exceeded 30 seconds: {url}')
                await asyncio.sleep(0.1)
    print('MCP 8010 and two A2A specialists 8001 are ready.')
except Exception:
    for process, log in services:
        process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            process.kill()
            process.wait()
        log.seek(0)
        print(log.read()[-4000:])
        log.close()
    services = []
    raise

## Inspect the Protocol Boundaries (20 min Including Startup)

The official A2A resolver reads each specialist's Agent Card. The official MCP client lists typed tools and invokes a hotel search as a short worked example. Inspect the tool name, arguments and returned dictionary, then adapt the invocation for Flight in task 1.

This probe performs one real hotel search, without booking or payment. `structured_content` is raw provider JSON; the specialist later applies W1 normalization and selection.

In [ ]:
from a2a.client import A2ACardResolver
from mcp import Client

async with httpx.AsyncClient() as http:
    for url in [a2a_coordinator.FLIGHT_URL, a2a_coordinator.HOTEL_URL]:
        card = await A2ACardResolver(httpx_client=http, base_url=url).get_agent_card()
        print(card.name, [(interface.url, interface.protocol_binding) for interface in card.supported_interfaces])
async with Client('http://127.0.0.1:8010/mcp') as client:
    tools = await client.list_tools()
    print('Official MCP tool list:', [tool.name for tool in tools.tools])
    arguments = {'request': request, 'occupancies': trip_planner.hotel_occupancies(request['adults_per_room'])}
    print('Tool request:', 'search_hotels', arguments)
    result = await client.call_tool('search_hotels', arguments)
    assert not result.is_error, result.content
    print('MCP response type:', type(result).__name__)
    print('Real hotel response keys:', list(result.structured_content))

## Run the Complete Path

After completing all three tasks, the next cell resolves both official A2A Agent Cards and requests the two selected quotes. Each specialist calls the real MCP service, which calls real Nuitee REST.

The teacher transport tests can pass before student tasks are complete. This end-to-end plan cannot. If a TODO is still present, the A2A failed-task message must reach the notebook.

In [ ]:
import json
import importlib
importlib.reload(a2a_coordinator)
plan = await a2a_coordinator.plan_trip(request)
print('Flight Agent response:', plan['flight'])
print('Hotel Agent response:', plan['hotel'])
print('Total SGD:', plan['total_cents'] / 100, 'Within budget:', plan['within_budget'])
(ROOT / 'trip_plan.json').write_text(json.dumps(plan, indent=2), encoding='utf-8')
print('Agent-built plan exported for Weeks 3-4.')

## Checks and Debrief (30 min)

Teacher transport checks: `python -m pytest tests/test_week2.py -m scaffold -q`.

Student acceptance checks: `python -m pytest tests/test_week2.py -m student -q`. These intentionally fail until the three tasks are implemented; no skip or xfail hides them. The tests verify reuse of Week 1 functions, not just matching totals.

Run both from the repository root. Unit tests use small test-only provider responses; they do not prove live availability. The preceding notebook plan is the separate live-provider check.

## Discuss

Which parts are A2A messages, which are MCP tool calls, and which are ordinary REST requests? Where does each boundary carry the same quote ID and currency? Why should the Hotel Agent not multiply the already-complete quote by five?

Changing transport must not change the Week 1 business functions. There is no payment authorization or booking in this lab.

## Shutdown

Run the next cell before restarting services or leaving the lab. It terminates only the processes started by this notebook. If a service failed, its temporary log is printed for diagnosis; never print API keys or authentication headers.

Weeks 3-4 read the exported `trip_plan.json` and do not need these services to remain running. No second travel planner is introduced.

In [ ]:
for process, log in services:
    failed = process.poll() not in [None, 0]
    process.terminate()
    try:
        process.wait(timeout=5)
    except subprocess.TimeoutExpired:
        process.kill()
        process.wait()
    if failed:
        log.seek(0)
        print(log.read()[-4000:])
    log.close()
services = []
print('Lab services stopped.')